# Drill-06 LLM Embedding Competition

## 한국어 영화 리뷰 감성 분석 — Hybrid (TF-IDF × Multilingual-E5 × KR-SBERT) Stacking

---

### Abstract

본 노트북은 한국어 영화 리뷰 (NSMC 계열, train 149,995 / test 49,997) 이진 감성 분류 (POSITIVE / NEGATIVE) 를 수행한다. 평가 지표는 **MCC (Matthews Correlation Coefficient)**.

**핵심 설계**:
1. **3가지 직교적 정보공간 결합**
   - Lexical: `TF-IDF (word 1-2) + (char_wb 2-5)`
   - Semantic (multilingual): `intfloat/multilingual-e5-base` (768d)
   - Semantic (korean-tuned): `snunlp/KR-SBERT-V40K-klueNLI-augSTS` (768d)
2. **5-fold stratified OOF training** — meta learner leakage 차단
3. **MCC-optimal threshold sweep** (OOF prediction에서만)
4. **Linear meta learner (LogReg L2)** — overfit-prone GBT 회피

**Drill-05 결과 비교**
| Stage | LB MCC |
|---|---|
| Drill-05 (TF-IDF stacking) | 0.7570 |
| **Drill-06 목표** | **0.770+** (개인 성장 +20점, 상위 6위 +30점) |

**핵심 철학**: validation MCC 가 아니라 **CV-LB gap 최소화**가 robust pipeline 의 본질.


---

## 1. Environment Setup

### 1.1 설계 결정

- **Colab T4 GPU 환경 가정** (drill-06-notice 권장). 15만 문장 인코딩을 분 단위로 완료.
- **재현성 보장**: 모든 stochastic 연산에 `SEED=42` 고정.
- **허용 라이브러리**: `sentence-transformers`, `scikit-learn`, `pandas`, `numpy`, `torch` (sentence-transformers 의존성).


In [ ]:
# Colab 환경 패키지 설치 (Colab에서만 실행, 로컬 무시)
!pip install -q sentence-transformers==2.7.0 scikit-learn==1.4.2 pandas numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.


In [ ]:
import os
import re
import gc
import time
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import matthews_corrcoef, confusion_matrix, classification_report
from scipy.sparse import csr_matrix, hstack

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU 확인 (drill-06 필수)
if torch.cuda.is_available():
    print(f"GPU OK: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: GPU not detected. Embedding will be slow.")

print(f"\nTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")


GPU OK: Tesla T4
  VRAM: 15.6 GB

Torch: 2.10.0+cu128
NumPy: 2.0.2
Pandas: 2.2.2


In [ ]:
# Colab Drive mount (Colab에서만)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/비정형 데이터 처리')
    CACHE_DIR = Path('/content/drive/MyDrive/비정형 데이터 처리/drill06_cache')
except (ImportError, ModuleNotFoundError):
    # 로컬 환경 (테스트용)
    DATA_DIR = Path('.')
    CACHE_DIR = Path('./drill06_cache')

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR : {DATA_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")


Mounted at /content/drive
DATA_DIR : /content/drive/MyDrive/비정형 데이터 처리
CACHE_DIR: /content/drive/MyDrive/비정형 데이터 처리/drill06_cache


---

## 2. Data Loading

### 2.1 설계 결정
- CSV 로딩 후 기본 무결성 점검 (shape, null, label 종류).
- text 컬럼 string cast — 일부 환경에서 NaN/숫자 mixed type 방지.


In [ ]:
train = pd.read_csv(DATA_DIR / 'public_train.csv', encoding='utf-8')
test  = pd.read_csv(DATA_DIR / 'public_test.csv',  encoding='utf-8')

# text 컬럼 안전화
train['text'] = train['text'].astype(str).fillna('')
test['text']  = test['text'].astype(str).fillna('')

print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")
print(f"Train cols : {list(train.columns)}")
print(f"Test  cols : {list(test.columns)}")
print()
print("Label values:", train['label'].unique())
print(train['label'].value_counts())

# label binarize: POSITIVE=1, NEGATIVE=0
LABEL_MAP = {'NEGATIVE': 0, 'POSITIVE': 1}
INV_LABEL = {0: 'NEGATIVE', 1: 'POSITIVE'}
train['y'] = train['label'].map(LABEL_MAP).astype(int)
y_full = train['y'].values
print(f"\ny=1 (POSITIVE) ratio: {y_full.mean():.4f}")


Train shape: (149995, 3)
Test  shape: (49997, 2)
Train cols : ['row_id', 'text', 'label']
Test  cols : ['row_id', 'text']

Label values: ['NEGATIVE' 'POSITIVE']
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64

y=1 (POSITIVE) ratio: 0.4988


---

## 3. Exploratory Data Analysis

### 3.1 검사 항목
- **Label 균형**: class imbalance 시 metric 해석 주의.
- **Length 분포**: max_len 설정 근거.
- **결측치**: 0건 가정 검증.


In [ ]:
# Length 통계 (char 단위)
lens_train = train['text'].str.len()
lens_test  = test['text'].str.len()

print("=== Train text length (chars) ===")
print(f"  mean={lens_train.mean():.1f}, median={lens_train.median():.0f}")
print(f"  p90={lens_train.quantile(0.90):.0f}, p99={lens_train.quantile(0.99):.0f}, max={lens_train.max()}")

print("\n=== Test text length (chars) ===")
print(f"  mean={lens_test.mean():.1f}, median={lens_test.median():.0f}")
print(f"  p90={lens_test.quantile(0.90):.0f}, p99={lens_test.quantile(0.99):.0f}, max={lens_test.max()}")

# 결측 점검
print(f"\nTrain nulls: {train.isnull().sum().to_dict()}")
print(f"Test  nulls: {test.isnull().sum().to_dict()}")

# 빈 문자열
n_empty_train = (train['text'].str.strip() == '').sum()
n_empty_test  = (test['text'].str.strip()  == '').sum()
print(f"\nEmpty strings - train: {n_empty_train}, test: {n_empty_test}")

# 샘플
print("\n=== 샘플 5건 ===")
for i in range(5):
    print(f"  [{train['label'].iloc[i]}] {train['text'].iloc[i][:80]}")


=== Train text length (chars) ===
  mean=35.2, median=27
  p90=75, p99=139, max=146

=== Test text length (chars) ===
  mean=35.3, median=27
  p90=76, p99=139, max=144

Train nulls: {'row_id': 0, 'text': 0, 'label': 0, 'y': 0}
Test  nulls: {'row_id': 0, 'text': 0}

Empty strings - train: 0, test: 0

=== 샘플 5건 ===
  [NEGATIVE] 아이 싱거워라...ㅠㅠ
  [POSITIVE] 히라이켄의 히토미오토지테 노래를 먼저 듣고 알게되어 보게 되었는데 영화 내용도 참 좋았고 그 이후에 듣는 히토미오토지테는 더더욱 와닿는다.
  [POSITIVE] 올 해 본 영화들 중 제일 맘에 든다
  [POSITIVE] 어릴적 로베르토 베니니 나온 실사 영화를 재밌게 봤었는데- 십여년이 지나 애니로 다시 만나니 감회가 새롭네요
  [NEGATIVE] 주드의팬이라면 대략 어의없을듯.사무엘은 네고시에이터의 변죽?


### 3.2 EDA 해석
- **클래스 균형** (~50:50) → `class_weight` 불필요. MCC 안정적.
- **짧은 리뷰** (평균 35자, p99~120자) → embedding max_len 128 충분.
- **결측치 0건** → imputation 불필요.

→ 모든 가정 충족. 본격 modeling 진행.


---

## 4. Preprocessing — 과전처리 금지

### 4.1 설계 원칙
- 영화 리뷰 평균 35자 — **모든 문자가 신호 후보**.
- 감성 marker (ㅋㅋ, ㅠㅠ, !!!, ♥, ★) 보존.
- pretrained embedding은 raw text에 학습됨 → 거의 무전처리가 inference 분포와 일치.

### 4.2 처리 내역
| 항목 | 처리 |
|---|---|
| URL `http(s)://...` | 제거 |
| HTML tag `<...>` | 제거 |
| Control char `\x00-\x1f` | 제거 |
| 연속 공백 | 1개로 normalize |
| 양끝 공백 | strip |
| **이모티콘/특수문자/반복문자** | **보존** |
| **대소문자** | **보존** (embedding은 case-aware, TF-IDF는 lowercase=True로 자체 처리) |


In [ ]:
RE_URL = re.compile(r'https?://\S+|www\.\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_CTRL = re.compile(r'[\x00-\x1f\x7f]')
RE_WS   = re.compile(r'\s+')

def clean_text(s: str) -> str:
    s = RE_URL.sub(' ', s)
    s = RE_HTML.sub(' ', s)
    s = RE_CTRL.sub(' ', s)
    s = RE_WS.sub(' ', s).strip()
    if not s:
        s = ' '  # embedding empty input 방지
    return s

t0 = time.time()
train['text_clean'] = train['text'].map(clean_text)
test['text_clean']  = test['text'].map(clean_text)
print(f"Preprocessing: {time.time()-t0:.1f}s")

# 정제 전/후 변화 점검
diffs = (train['text'] != train['text_clean']).sum()
print(f"변경된 train 샘플: {diffs:,} / {len(train):,} ({diffs/len(train)*100:.2f}%)")

# 샘플
print("\n=== 정제 전/후 샘플 ===")
sample_idx = train['text'].str.contains(r'http|<|[\x00-\x1f]', regex=True, na=False)
if sample_idx.sum() > 0:
    for i in train.index[sample_idx][:3]:
        print(f"  BEFORE: {train['text'].iloc[i][:80]!r}")
        print(f"  AFTER : {train['text_clean'].iloc[i][:80]!r}")
        print()
else:
    print("  (URL/HTML/ctrl 포함 샘플 없음)")


Preprocessing: 1.5s
변경된 train 샘플: 200 / 149,995 (0.13%)

=== 정제 전/후 샘플 ===
  BEFORE: '스토리도 탄탄하고 이윤성과 김영주검사의 심리싸움(?)도 정말 기대하게 만들어용♥ 김나나하고 이윤성 잘 어울려><'
  AFTER : '스토리도 탄탄하고 이윤성과 김영주검사의 심리싸움(?)도 정말 기대하게 만들어용♥ 김나나하고 이윤성 잘 어울려><'

  BEFORE: '><정지훈이당 ㅜㅜ보고싶다정지훈 ㅠㅠ'
  AFTER : '><정지훈이당 ㅜㅜ보고싶다정지훈 ㅠㅠ'

  BEFORE: '<진심>이란 게 어떤 건지 제대로 보여주는 영화. 98%가 이슬람교인 <이란>이기에 더 절묘해진 영화.'
  AFTER : '이란 게 어떤 건지 제대로 보여주는 영화. 98%가 이슬람교인 이기에 더 절묘해진 영화.'



---

## 5. Embedding Generation — 2-model Strategy

### 5.1 모델 선정 근거
| 모델 | dim | 선정 이유 |
|---|---|---|
| `intfloat/multilingual-e5-base` | 768 | retrieval-tuned, paraphrase robustness. 다국어 corpus 학습으로 generalization 강함. |
| `snunlp/KR-SBERT-V40K-klueNLI-augSTS` | 768 | 한국어 NLI/STS fine-tuned. e5와 학습 paradigm/domain 다름 → **diversity 확보**. |

**채택 이유**: 두 모델은 학습 task (retrieval vs NLI) 와 학습 데이터 분포가 달라 **decision boundary diversity** 가 확보됨. 동일 family (e5-base + e5-large) 는 correlation 너무 높아 ensemble 효과 미미.

### 5.2 E5 모델 prefix 규칙
- E5 계열은 `"query: "` 또는 `"passage: "` prefix 필요. 분류 task 에는 `"query: "` 사용.
- KR-SBERT는 prefix 불필요.

### 5.3 캐싱
- 인코딩 1회 비용 큼 → numpy `.npy`로 disk cache. 재실행 시 skip.


In [ ]:
from sentence_transformers import SentenceTransformer

EMB_MODELS = {
    'e5':       'intfloat/multilingual-e5-base',
    'krsbert':  'snunlp/KR-SBERT-V40K-klueNLI-augSTS',
}

def encode_with_cache(model_key: str, model_name: str, texts_train, texts_test, prefix=''):
    cache_train = CACHE_DIR / f'emb_{model_key}_train.npy'
    cache_test  = CACHE_DIR / f'emb_{model_key}_test.npy'

    if cache_train.exists() and cache_test.exists():
        print(f"[{model_key}] cache hit → load")
        return np.load(cache_train), np.load(cache_test)

    print(f"[{model_key}] loading model: {model_name}")
    model = SentenceTransformer(model_name, device='cuda' if torch.cuda.is_available() else 'cpu')
    model.max_seq_length = 128

    def _encode(texts, desc):
        inputs = [prefix + t for t in texts] if prefix else list(texts)
        t0 = time.time()
        emb = model.encode(
            inputs,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,   # L2 norm = 1 → cosine geometry
        )
        print(f"  [{desc}] {emb.shape} in {time.time()-t0:.1f}s")
        return emb.astype(np.float32)

    emb_tr = _encode(texts_train, 'train')
    emb_te = _encode(texts_test,  'test')

    np.save(cache_train, emb_tr)
    np.save(cache_test,  emb_te)
    print(f"  cached → {cache_train.name}, {cache_test.name}")

    # GPU 메모리 해제
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return emb_tr, emb_te

# E5 (with query: prefix)
emb_e5_train, emb_e5_test = encode_with_cache(
    'e5', EMB_MODELS['e5'],
    train['text_clean'].tolist(), test['text_clean'].tolist(),
    prefix='query: '
)

# KR-SBERT (no prefix)
emb_kr_train, emb_kr_test = encode_with_cache(
    'krsbert', EMB_MODELS['krsbert'],
    train['text_clean'].tolist(), test['text_clean'].tolist(),
    prefix=''
)

print(f"\nE5      : train {emb_e5_train.shape}, test {emb_e5_test.shape}")
print(f"KR-SBERT: train {emb_kr_train.shape}, test {emb_kr_test.shape}")


[e5] loading model: intfloat/multilingual-e5-base


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

  [train] (149995, 768) in 290.6s


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

  [test] (49997, 768) in 96.1s
  cached → emb_krsbert_train.npy, emb_krsbert_test.npy

E5      : train (149995, 768), test (49997, 768)
KR-SBERT: train (149995, 768), test (49997, 768)


---

## 6. Hand-crafted Meta Features

### 6.1 설계 근거
TF-IDF / embedding 이 놓치는 **문서 구조 정보** 보강. Drill-05 에서 +0.001~+0.003 MCC 기여 확인됨.

### 6.2 Feature 목록 (10개)
| Feature | 의미 |
|---|---|
| len_chars | 문자 수 |
| log_len | log(len+1) — 길이 outlier 완화 |
| n_excl | `!` 개수 |
| n_quest | `?` 개수 |
| n_dot_runs | `...` 개수 |
| n_kkk | `ㅋ` 반복 (POSITIVE marker) |
| n_ttt | `ㅠ/ㅜ` 반복 (보통 NEGATIVE, 가끔 강조) |
| n_hhh | `ㅎ` 반복 (POSITIVE marker) |
| n_negation | 안/못/않/없/아니 출현 횟수 |
| n_intensifier | 정말/진짜/매우/너무/완전 |


In [ ]:
RE_KKK = re.compile(r'ㅋ+')
RE_TTT = re.compile(r'[ㅠㅜ]+')
RE_HHH = re.compile(r'ㅎ+')
RE_DOT = re.compile(r'\.{2,}')
RE_NEG = re.compile(r'(안 |못 |않|없|아니)')
RE_INT = re.compile(r'(정말|진짜|매우|너무|아주|완전)')

def meta_features(texts):
    arr = np.zeros((len(texts), 10), dtype=np.float32)
    for i, s in enumerate(texts):
        L = len(s)
        arr[i,0] = L
        arr[i,1] = np.log1p(L)
        arr[i,2] = s.count('!')
        arr[i,3] = s.count('?')
        arr[i,4] = len(RE_DOT.findall(s))
        arr[i,5] = sum(len(m) for m in RE_KKK.findall(s))
        arr[i,6] = sum(len(m) for m in RE_TTT.findall(s))
        arr[i,7] = sum(len(m) for m in RE_HHH.findall(s))
        arr[i,8] = len(RE_NEG.findall(s))
        arr[i,9] = len(RE_INT.findall(s))
    return arr

META_COLS = ['len_chars','log_len','n_excl','n_quest','n_dot_runs',
             'n_kkk','n_ttt','n_hhh','n_negation','n_intensifier']

t0 = time.time()
meta_train = meta_features(train['text_clean'].tolist())
meta_test  = meta_features(test['text_clean'].tolist())
print(f"Meta features: train {meta_train.shape}, test {meta_test.shape} ({time.time()-t0:.1f}s)")

# 빠른 sanity: POSITIVE vs NEGATIVE에서 feature 평균 비교
print("\n=== Meta feature 평균 (POS vs NEG) ===")
for j, name in enumerate(META_COLS):
    pos = meta_train[y_full==1, j].mean()
    neg = meta_train[y_full==0, j].mean()
    print(f"  {name:15s}: POS={pos:7.3f}  NEG={neg:7.3f}  Δ={pos-neg:+.3f}")


Meta features: train (149995, 10), test (49997, 10) (1.9s)

=== Meta feature 평균 (POS vs NEG) ===
  len_chars      : POS= 34.573  NEG= 35.780  Δ=-1.207
  log_len        : POS=  3.291  NEG=  3.317  Δ=-0.026
  n_excl         : POS=  0.314  NEG=  0.111  Δ=+0.203
  n_quest        : POS=  0.070  NEG=  0.175  Δ=-0.105
  n_dot_runs     : POS=  0.340  NEG=  0.480  Δ=-0.140
  n_kkk          : POS=  0.242  NEG=  0.228  Δ=+0.013
  n_ttt          : POS=  0.113  NEG=  0.047  Δ=+0.067
  n_hhh          : POS=  0.066  NEG=  0.014  Δ=+0.052
  n_negation     : POS=  0.174  NEG=  0.336  Δ=-0.162
  n_intensifier  : POS=  0.270  NEG=  0.201  Δ=+0.068


---

## 7. Stratified 5-fold OOF Base Models

### 7.1 왜 OOF 인가
- Meta learner가 base proba 를 입력 받음.
- base 가 **자기 학습 데이터를 다시 예측** 하면 proba 가 비현실적으로 좋음 (train MCC 0.95+) → meta 가 이를 신뢰하고 overfit.
- 따라서 base proba 는 반드시 **fold 외부 데이터로 학습한 모델의 in-fold prediction**.

### 7.2 4-Base Model 구성
1. `tfidf_word` : TF-IDF (1,2) word + LogReg ElasticNet
2. `tfidf_char` : TF-IDF char_wb (2,5) + LogReg ElasticNet
3. `emb_e5`     : multilingual-e5-base + StandardScaler + LogReg L2
4. `emb_kr`     : KR-SBERT + StandardScaler + LogReg L2

각 fold마다 vectorizer/scaler 새로 fit (leakage 차단). 모델 hyperparameter는 Drill-05 검증된 값 + embedding 모델은 C=2.0 (sklearn default 보다 약간 강한 fit, regularization은 L2 유지).


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

N_TRAIN = len(train)
N_TEST  = len(test)

# OOF prediction storage
oof = {
    'tfidf_word': np.zeros(N_TRAIN, dtype=np.float32),
    'tfidf_char': np.zeros(N_TRAIN, dtype=np.float32),
    'emb_e5':     np.zeros(N_TRAIN, dtype=np.float32),
    'emb_kr':     np.zeros(N_TRAIN, dtype=np.float32),
}
# Test prediction (fold 평균)
test_pred = {k: np.zeros(N_TEST, dtype=np.float32) for k in oof.keys()}

texts_full  = train['text_clean'].values
texts_test  = test['text_clean'].values

# fold 별 train/val MCC 기록
fold_log = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.arange(N_TRAIN), y_full)):
    t_fold = time.time()
    print(f"\n=========== FOLD {fold+1}/5 ===========")
    print(f"  train: {len(tr_idx):,} / val: {len(va_idx):,}")
    y_tr, y_va = y_full[tr_idx], y_full[va_idx]

    # ----- (1) TF-IDF word -----
    vec_w = TfidfVectorizer(
        ngram_range=(1,2), min_df=3, max_df=0.95,
        sublinear_tf=True, lowercase=True,
    )
    Xw_tr = vec_w.fit_transform(texts_full[tr_idx])
    Xw_va = vec_w.transform(texts_full[va_idx])
    Xw_te = vec_w.transform(texts_test)
    clf_w = LogisticRegression(
        penalty='elasticnet', l1_ratio=0.15, C=2.0, solver='saga',
        max_iter=200, random_state=SEED, n_jobs=-1,
    )
    clf_w.fit(Xw_tr, y_tr)
    p_va = clf_w.predict_proba(Xw_va)[:,1]
    p_te = clf_w.predict_proba(Xw_te)[:,1]
    oof['tfidf_word'][va_idx] = p_va
    test_pred['tfidf_word'] += p_te / 5.0
    tr_mcc = matthews_corrcoef(y_tr, (clf_w.predict_proba(Xw_tr)[:,1] >= 0.5).astype(int))
    va_mcc = matthews_corrcoef(y_va, (p_va >= 0.5).astype(int))
    print(f"  [tfidf_word] feat={Xw_tr.shape[1]:>7,} train MCC={tr_mcc:.4f}  val MCC={va_mcc:.4f}")
    fold_log.append({'fold':fold,'model':'tfidf_word','train_mcc':tr_mcc,'val_mcc':va_mcc})
    del vec_w, clf_w, Xw_tr, Xw_va, Xw_te; gc.collect()

    # ----- (2) TF-IDF char_wb -----
    vec_c = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2,5), min_df=5, max_df=0.95,
        sublinear_tf=True, lowercase=True,
    )
    Xc_tr = vec_c.fit_transform(texts_full[tr_idx])
    Xc_va = vec_c.transform(texts_full[va_idx])
    Xc_te = vec_c.transform(texts_test)
    clf_c = LogisticRegression(
        penalty='elasticnet', l1_ratio=0.15, C=2.0, solver='saga',
        max_iter=200, random_state=SEED, n_jobs=-1,
    )
    clf_c.fit(Xc_tr, y_tr)
    p_va = clf_c.predict_proba(Xc_va)[:,1]
    p_te = clf_c.predict_proba(Xc_te)[:,1]
    oof['tfidf_char'][va_idx] = p_va
    test_pred['tfidf_char'] += p_te / 5.0
    tr_mcc = matthews_corrcoef(y_tr, (clf_c.predict_proba(Xc_tr)[:,1] >= 0.5).astype(int))
    va_mcc = matthews_corrcoef(y_va, (p_va >= 0.5).astype(int))
    print(f"  [tfidf_char] feat={Xc_tr.shape[1]:>7,} train MCC={tr_mcc:.4f}  val MCC={va_mcc:.4f}")
    fold_log.append({'fold':fold,'model':'tfidf_char','train_mcc':tr_mcc,'val_mcc':va_mcc})
    del vec_c, clf_c, Xc_tr, Xc_va, Xc_te; gc.collect()

    # ----- (3) Embedding e5 -----
    sc_e5 = StandardScaler()
    Xe_tr = sc_e5.fit_transform(emb_e5_train[tr_idx])
    Xe_va = sc_e5.transform(emb_e5_train[va_idx])
    Xe_te = sc_e5.transform(emb_e5_test)
    clf_e = LogisticRegression(
        penalty='l2', C=2.0, solver='lbfgs',
        max_iter=500, random_state=SEED, n_jobs=-1,
    )
    clf_e.fit(Xe_tr, y_tr)
    p_va = clf_e.predict_proba(Xe_va)[:,1]
    p_te = clf_e.predict_proba(Xe_te)[:,1]
    oof['emb_e5'][va_idx] = p_va
    test_pred['emb_e5'] += p_te / 5.0
    tr_mcc = matthews_corrcoef(y_tr, (clf_e.predict_proba(Xe_tr)[:,1] >= 0.5).astype(int))
    va_mcc = matthews_corrcoef(y_va, (p_va >= 0.5).astype(int))
    print(f"  [emb_e5    ] dim ={Xe_tr.shape[1]:>7,} train MCC={tr_mcc:.4f}  val MCC={va_mcc:.4f}")
    fold_log.append({'fold':fold,'model':'emb_e5','train_mcc':tr_mcc,'val_mcc':va_mcc})
    del sc_e5, clf_e, Xe_tr, Xe_va, Xe_te; gc.collect()

    # ----- (4) Embedding KR-SBERT -----
    sc_kr = StandardScaler()
    Xk_tr = sc_kr.fit_transform(emb_kr_train[tr_idx])
    Xk_va = sc_kr.transform(emb_kr_train[va_idx])
    Xk_te = sc_kr.transform(emb_kr_test)
    clf_k = LogisticRegression(
        penalty='l2', C=2.0, solver='lbfgs',
        max_iter=500, random_state=SEED, n_jobs=-1,
    )
    clf_k.fit(Xk_tr, y_tr)
    p_va = clf_k.predict_proba(Xk_va)[:,1]
    p_te = clf_k.predict_proba(Xk_te)[:,1]
    oof['emb_kr'][va_idx] = p_va
    test_pred['emb_kr'] += p_te / 5.0
    tr_mcc = matthews_corrcoef(y_tr, (clf_k.predict_proba(Xk_tr)[:,1] >= 0.5).astype(int))
    va_mcc = matthews_corrcoef(y_va, (p_va >= 0.5).astype(int))
    print(f"  [emb_kr    ] dim ={Xk_tr.shape[1]:>7,} train MCC={tr_mcc:.4f}  val MCC={va_mcc:.4f}")
    fold_log.append({'fold':fold,'model':'emb_kr','train_mcc':tr_mcc,'val_mcc':va_mcc})
    del sc_kr, clf_k, Xk_tr, Xk_va, Xk_te; gc.collect()

    print(f"  fold {fold+1} elapsed: {time.time()-t_fold:.1f}s")

# 전체 OOF MCC (각 base 단독)
print("\n=== OOF MCC by base model (threshold=0.5) ===")
for k, p in oof.items():
    mcc = matthews_corrcoef(y_full, (p >= 0.5).astype(int))
    print(f"  {k:12s}: OOF MCC = {mcc:.4f}")



=========== FOLD 1/5 ===========
  train: 119,996 / val: 29,999
  [tfidf_word] feat= 50,565 train MCC=0.7569  val MCC=0.6288
  [tfidf_char] feat=207,339 train MCC=0.8351  val MCC=0.7406
  [emb_e5    ] dim =    768 train MCC=0.6670  val MCC=0.6633
  [emb_kr    ] dim =    768 train MCC=0.7120  val MCC=0.7132
  fold 1 elapsed: 2623.7s

=========== FOLD 2/5 ===========
  train: 119,996 / val: 29,999
  [tfidf_word] feat= 50,510 train MCC=0.7574  val MCC=0.6157
  [tfidf_char] feat=207,194 train MCC=0.8374  val MCC=0.7312
  [emb_e5    ] dim =    768 train MCC=0.6705  val MCC=0.6492
  [emb_kr    ] dim =    768 train MCC=0.7155  val MCC=0.7006
  fold 2 elapsed: 2592.0s

=========== FOLD 3/5 ===========
  train: 119,996 / val: 29,999
  [tfidf_word] feat= 50,596 train MCC=0.7563  val MCC=0.6252
  [tfidf_char] feat=207,632 train MCC=0.8358  val MCC=0.7343
  [emb_e5    ] dim =    768 train MCC=0.6685  val MCC=0.6589
  [emb_kr    ] dim =    768 train MCC=0.7143  val MCC=0.7047
  fold 3 elapsed: 255

---

## 8. Base Model Diversity 분석

### 8.1 OOF Proba Correlation
- 0.95 이상 → 사실상 같은 정보 → ensemble 효과 미미.
- 0.5~0.8 → 상호보완적, ensemble 효과 큼 (이 범위 목표).

### 8.2 분석 시점
- 의도적으로 OOF prediction (in-fold) 으로 계산 — fold 외부 데이터로 학습된 모델의 unbiased prediction.


In [ ]:
# 4 base의 OOF proba dataframe
oof_df = pd.DataFrame({k: v for k, v in oof.items()})
print("=== OOF proba Pearson correlation ===")
corr = oof_df.corr()
print(corr.round(3).to_string())

# 두 그룹 정의: lexical (tfidf_word, tfidf_char) vs semantic (emb_e5, emb_kr)
print("\n=== Cross-group correlation ===")
for lex in ['tfidf_word','tfidf_char']:
    for sem in ['emb_e5','emb_kr']:
        print(f"  {lex:12s} × {sem:8s} = {corr.loc[lex,sem]:.3f}")

print("\n=== Intra-group correlation ===")
print(f"  tfidf_word × tfidf_char = {corr.loc['tfidf_word','tfidf_char']:.3f}")
print(f"  emb_e5     × emb_kr     = {corr.loc['emb_e5','emb_kr']:.3f}")


=== OOF proba Pearson correlation ===
            tfidf_word  tfidf_char  emb_e5  emb_kr
tfidf_word       1.000       0.880   0.754   0.771
tfidf_char       0.880       1.000   0.844   0.865
emb_e5           0.754       0.844   1.000   0.841
emb_kr           0.771       0.865   0.841   1.000

=== Cross-group correlation ===
  tfidf_word   × emb_e5   = 0.754
  tfidf_word   × emb_kr   = 0.771
  tfidf_char   × emb_e5   = 0.844
  tfidf_char   × emb_kr   = 0.865

=== Intra-group correlation ===
  tfidf_word × tfidf_char = 0.880
  emb_e5     × emb_kr     = 0.841


### 8.3 해석 가이드
- **intra-group (TF-IDF × TF-IDF) ~ 0.8+**: 같은 lexical 정보. 그래도 char_wb 가 typo 잡으므로 유지.
- **intra-group (emb_e5 × emb_kr) ~ 0.7+**: 같은 paradigm (transformer) 이라 약간 높음. 학습 corpus 다르므로 보완성 있음.
- **cross-group (lexical × semantic) ~ 0.5~0.6**: **핵심 다양성 source**. ensemble 효과 대부분 여기서 나옴.


---

## 9. Meta Stacking — LogReg L2

### 9.1 설계
- **Input**: 4 base OOF proba + 10 meta features = 14d
- **Model**: LogReg L2 (C=1.0) — Drill-05 검증된 안정적 meta learner
- **Why not GBT**: Drill-05 에서 HistGBT는 Val 0.7748 → LB 0.7500 (-0.025). 30K × 14 환경에서 비선형 표현력이 sample noise 학습.
- **Why L2 (not ElasticNet)**: 14d 저차원에서 L1 sparsity 불필요.

### 9.2 학습 데이터
- meta learner는 **OOF prediction** 으로 학습. 이미 leakage-free 보장.
- 별도 holdout 불필요 (5-fold가 곧 holdout).


In [ ]:
# Meta feature matrix: 4 base proba + 10 hand-crafted
def build_meta_input(oof_dict, meta_arr):
    base_arr = np.column_stack([oof_dict[k] for k in ['tfidf_word','tfidf_char','emb_e5','emb_kr']])
    return np.concatenate([base_arr, meta_arr], axis=1)

X_meta_train = build_meta_input(oof, meta_train)
print(f"Meta input shape: {X_meta_train.shape}")

meta_scaler = StandardScaler()
X_meta_train_sc = meta_scaler.fit_transform(X_meta_train)

meta_clf = LogisticRegression(
    penalty='l2', C=1.0, solver='lbfgs',
    max_iter=1000, random_state=SEED, n_jobs=-1,
)
meta_clf.fit(X_meta_train_sc, y_full)

p_meta_oof = meta_clf.predict_proba(X_meta_train_sc)[:,1]
# 주의: 이 p_meta_oof 는 meta_clf 가 OOF data 로 학습 후 같은 data 로 예측 → meta 의 train fit.
# 진정한 meta-OOF 를 보려면 한 단계 더 nested CV 필요하나, meta input 자체가 이미 OOF.
# Drill-05 검증: 이 방식이 LB transfer 우수.

print("\n=== OOF MCC 비교 (threshold=0.5) ===")
print(f"  단순 평균 (4 base)         : {matthews_corrcoef(y_full, ((oof['tfidf_word']+oof['tfidf_char']+oof['emb_e5']+oof['emb_kr'])/4 >= 0.5).astype(int)):.4f}")
print(f"  Meta LogReg (stacking)     : {matthews_corrcoef(y_full, (p_meta_oof >= 0.5).astype(int)):.4f}")

# Meta 계수 inspection
print("\n=== Meta LogReg 계수 (절댓값 기준 정렬) ===")
feat_names = ['p_tfidf_word','p_tfidf_char','p_emb_e5','p_emb_kr'] + META_COLS
coef = meta_clf.coef_[0]
for name, c in sorted(zip(feat_names, coef), key=lambda x: -abs(x[1])):
    print(f"  {name:15s}: {c:+.4f}")


Meta input shape: (149995, 14)

=== OOF MCC 비교 (threshold=0.5) ===
  단순 평균 (4 base)         : 0.7602
  Meta LogReg (stacking)     : 0.7636

=== Meta LogReg 계수 (절댓값 기준 정렬) ===
  p_tfidf_char   : +1.2896
  p_emb_kr       : +0.9236
  p_tfidf_word   : +0.4265
  p_emb_e5       : +0.4023
  log_len        : +0.0621
  len_chars      : -0.0604
  n_negation     : +0.0429
  n_dot_runs     : +0.0336
  n_excl         : -0.0332
  n_quest        : +0.0321
  n_intensifier  : -0.0169
  n_kkk          : -0.0077
  n_hhh          : -0.0028
  n_ttt          : +0.0021


---

## 10. MCC-optimal Threshold Sweep

### 10.1 왜 threshold 0.5 가 항상 최적이 아닌가
- ensemble 후 probability calibration 손상.
- meta learner 의 sigmoid output 분포가 systemic bias 가질 수 있음.
- MCC 는 클래스별 marginal precision/recall 에 모두 민감 → 작은 threshold shift 가 ±0.005~0.015 MCC 변동.

### 10.2 전략
- 0.30~0.70, 0.005 step → 81 candidates.
- **OOF prediction에서만** sweep — test 정보 절대 미사용.
- argmax 와 plateau 동시 검토. plateau 좁으면 overfit risk → 0.5 fallback.

### 10.3 Safety Net
- τ* 와 τ=0.5 의 MCC 차이가 0.002 미만이면 → τ=0.5 사용.
- 즉, 의미 있는 개선이 있을 때만 τ 이동.


In [ ]:
thresholds = np.linspace(0.30, 0.70, 81)
mccs = np.array([matthews_corrcoef(y_full, (p_meta_oof >= t).astype(int)) for t in thresholds])

best_idx = int(np.argmax(mccs))
tau_star = float(thresholds[best_idx])
mcc_star = float(mccs[best_idx])
mcc_default = float(matthews_corrcoef(y_full, (p_meta_oof >= 0.5).astype(int)))

print(f"  argmax τ* = {tau_star:.3f}  (MCC = {mcc_star:.4f})")
print(f"  default τ = 0.500   (MCC = {mcc_default:.4f})")
print(f"  gain      = {mcc_star - mcc_default:+.4f}")

# plateau 분석 (±0.005 범위에서 MCC 변동)
plateau_lo = thresholds[max(0, best_idx-3)]
plateau_hi = thresholds[min(len(thresholds)-1, best_idx+3)]
plateau_min_mcc = mccs[max(0, best_idx-3):min(len(thresholds), best_idx+4)].min()
print(f"  plateau τ in [{plateau_lo:.3f}, {plateau_hi:.3f}], min MCC = {plateau_min_mcc:.4f}")

# Safety net
GAIN_THRESHOLD = 0.002
if mcc_star - mcc_default < GAIN_THRESHOLD:
    print(f"  → gain < {GAIN_THRESHOLD}, falling back to τ=0.500 (transfer risk avoidance)")
    tau_final = 0.5
else:
    print(f"  → adopting τ*={tau_star:.3f}")
    tau_final = tau_star

print(f"\n  FINAL THRESHOLD = {tau_final:.3f}")


  argmax τ* = 0.520  (MCC = 0.7638)
  default τ = 0.500   (MCC = 0.7636)
  gain      = +0.0002
  plateau τ in [0.505, 0.535], min MCC = 0.7630
  → gain < 0.002, falling back to τ=0.500 (transfer risk avoidance)

  FINAL THRESHOLD = 0.500


In [ ]:
# Threshold curve plot
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(thresholds, mccs, 'b-', linewidth=2, label='OOF MCC')
ax.axvline(tau_star, color='r', linestyle='--', alpha=0.7, label=f'τ* = {tau_star:.3f}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='τ = 0.500')
ax.axhline(mcc_star, color='r', linestyle=':', alpha=0.3)
ax.set_xlabel('Threshold τ')
ax.set_ylabel('MCC (OOF)')
ax.set_title(f'MCC vs Threshold — best τ*={tau_star:.3f}, MCC={mcc_star:.4f}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CACHE_DIR / 'threshold_curve.png', dpi=100)
plt.show()
print(f"saved: {CACHE_DIR / 'threshold_curve.png'}")


saved: /content/drive/MyDrive/비정형 데이터 처리/drill06_cache/threshold_curve.png


---

## 11. Validation Report

### 11.1 Confusion Matrix + classification report (OOF, τ_final)


In [ ]:
y_pred_oof = (p_meta_oof >= tau_final).astype(int)
oof_mcc_final = matthews_corrcoef(y_full, y_pred_oof)

cm = confusion_matrix(y_full, y_pred_oof, labels=[0,1])
print(f"=== Confusion Matrix (OOF, τ={tau_final}) ===")
print(f"                pred_NEG  pred_POS")
print(f"  true_NEG      {cm[0,0]:>8,}  {cm[0,1]:>8,}")
print(f"  true_POS      {cm[1,0]:>8,}  {cm[1,1]:>8,}")

tn, fp, fn, tp = cm.ravel()
precision_pos = tp / (tp + fp) if (tp+fp) else 0
recall_pos    = tp / (tp + fn) if (tp+fn) else 0
precision_neg = tn / (tn + fn) if (tn+fn) else 0
recall_neg    = tn / (tn + fp) if (tn+fp) else 0

print(f"\n=== Per-class metrics ===")
print(f"  POSITIVE precision={precision_pos:.4f}  recall={recall_pos:.4f}")
print(f"  NEGATIVE precision={precision_neg:.4f}  recall={recall_neg:.4f}")
print(f"\n  OOF MCC (FINAL) = {oof_mcc_final:.4f}")


=== Confusion Matrix (OOF, τ=0.5) ===
                pred_NEG  pred_POS
  true_NEG        66,692     8,478
  true_POS         9,256    65,569

=== Per-class metrics ===
  POSITIVE precision=0.8855  recall=0.8763
  NEGATIVE precision=0.8781  recall=0.8872

  OOF MCC (FINAL) = 0.7636


### 11.2 Per-base OOF MCC 비교 (ablation table)


In [ ]:
print("=== Ablation table ===")
print(f"{'Stage':40s} {'OOF MCC':>10s}")
print("-"*52)
print(f"{'(A) tfidf_word only':40s} {matthews_corrcoef(y_full, (oof['tfidf_word']>=0.5).astype(int)):10.4f}")
print(f"{'(A) tfidf_char only':40s} {matthews_corrcoef(y_full, (oof['tfidf_char']>=0.5).astype(int)):10.4f}")
print(f"{'(B) emb_e5 only':40s} {matthews_corrcoef(y_full, (oof['emb_e5']>=0.5).astype(int)):10.4f}")
print(f"{'(B) emb_kr only':40s} {matthews_corrcoef(y_full, (oof['emb_kr']>=0.5).astype(int)):10.4f}")
print(f"{'(D) 4-base simple average':40s} {matthews_corrcoef(y_full, ((oof['tfidf_word']+oof['tfidf_char']+oof['emb_e5']+oof['emb_kr'])/4 >= 0.5).astype(int)):10.4f}")
print(f"{'(E) Meta stacking + τ=0.5':40s} {mcc_default:10.4f}")
print(f"{'(F) Meta stacking + τ*':40s} {oof_mcc_final:10.4f}")


=== Ablation table ===
Stage                                       OOF MCC
----------------------------------------------------
(A) tfidf_word only                          0.6232
(A) tfidf_char only                          0.7366
(B) emb_e5 only                              0.6608
(B) emb_kr only                              0.7072
(D) 4-base simple average                    0.7602
(E) Meta stacking + τ=0.5                    0.7636
(F) Meta stacking + τ*                       0.7636


---

## 12. Final Retraining on Full Train

### 12.1 전략
- Base 모델 4개를 **train 전체 (149,995)** 로 재학습.
- Meta learner 는 **OOF 로 학습한 것 그대로** 재사용. 이유: 재학습 시 leakage-free input 만들 수 없음.
- Test prediction = 재학습된 base 4개로 test 예측 → meta_clf 적용 → τ_final 이진화.


In [ ]:
# 4 base 모델 full train 재학습
t0 = time.time()

# (1) TF-IDF word
vec_w_full = TfidfVectorizer(
    ngram_range=(1,2), min_df=3, max_df=0.95,
    sublinear_tf=True, lowercase=True,
)
Xw_full = vec_w_full.fit_transform(texts_full)
Xw_test_final = vec_w_full.transform(texts_test)
clf_w_full = LogisticRegression(
    penalty='elasticnet', l1_ratio=0.15, C=2.0, solver='saga',
    max_iter=200, random_state=SEED, n_jobs=-1,
)
clf_w_full.fit(Xw_full, y_full)
p_test_w = clf_w_full.predict_proba(Xw_test_final)[:,1]
print(f"  [tfidf_word full] feat={Xw_full.shape[1]:,} done")

# (2) TF-IDF char
vec_c_full = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2,5), min_df=5, max_df=0.95,
    sublinear_tf=True, lowercase=True,
)
Xc_full = vec_c_full.fit_transform(texts_full)
Xc_test_final = vec_c_full.transform(texts_test)
clf_c_full = LogisticRegression(
    penalty='elasticnet', l1_ratio=0.15, C=2.0, solver='saga',
    max_iter=200, random_state=SEED, n_jobs=-1,
)
clf_c_full.fit(Xc_full, y_full)
p_test_c = clf_c_full.predict_proba(Xc_test_final)[:,1]
print(f"  [tfidf_char full] feat={Xc_full.shape[1]:,} done")

del Xw_full, Xc_full; gc.collect()

# (3) e5
sc_e5_full = StandardScaler()
Xe_full = sc_e5_full.fit_transform(emb_e5_train)
Xe_test_final = sc_e5_full.transform(emb_e5_test)
clf_e_full = LogisticRegression(penalty='l2', C=2.0, solver='lbfgs',
    max_iter=500, random_state=SEED, n_jobs=-1)
clf_e_full.fit(Xe_full, y_full)
p_test_e = clf_e_full.predict_proba(Xe_test_final)[:,1]
print(f"  [emb_e5 full]    done")

# (4) KR-SBERT
sc_kr_full = StandardScaler()
Xk_full = sc_kr_full.fit_transform(emb_kr_train)
Xk_test_final = sc_kr_full.transform(emb_kr_test)
clf_k_full = LogisticRegression(penalty='l2', C=2.0, solver='lbfgs',
    max_iter=500, random_state=SEED, n_jobs=-1)
clf_k_full.fit(Xk_full, y_full)
p_test_k = clf_k_full.predict_proba(Xk_test_final)[:,1]
print(f"  [emb_kr full]    done")

del Xe_full, Xk_full; gc.collect()

print(f"\nfull retrain: {time.time()-t0:.1f}s")


  [tfidf_word full] feat=62,647 done
  [tfidf_char full] feat=248,383 done
  [emb_e5 full]    done
  [emb_kr full]    done

full retrain: 3879.9s


---

## 13. Test Inference

### 13.1 Pipeline
1. Test 의 4 base proba 결합 (`p_test_w, p_test_c, p_test_e, p_test_k`).
2. meta features (test) 추출.
3. meta_scaler (train fit) 적용.
4. meta_clf 로 final proba 산출.
5. τ_final 적용 → POSITIVE / NEGATIVE.


In [ ]:
# Test 의 meta input 구성
X_meta_test = np.column_stack([p_test_w, p_test_c, p_test_e, p_test_k, *[meta_test[:,j] for j in range(meta_test.shape[1])]])
print(f"Test meta input shape: {X_meta_test.shape}")

X_meta_test_sc = meta_scaler.transform(X_meta_test)
p_meta_test = meta_clf.predict_proba(X_meta_test_sc)[:,1]

y_test_pred = (p_meta_test >= tau_final).astype(int)
labels_test = [INV_LABEL[int(v)] for v in y_test_pred]

# 분포 점검 (train과 비슷해야 정상)
from collections import Counter
dist = Counter(labels_test)
print(f"\nTest 예측 분포: {dict(dist)}")
print(f"  POSITIVE ratio: {dist['POSITIVE']/len(labels_test):.4f} (train: {y_full.mean():.4f})")


---

## 14. Submission CSV
- 파일명: `submission_llm_202431626_김지우.csv`
- 컬럼 순서: `row_id, pred_label`


In [ ]:
SUBMISSION_PATH = DATA_DIR / 'submission_llm_202431626_김지우.csv'

submission = pd.DataFrame({
    'row_id': test['row_id'].values,
    'pred_label': labels_test,
})

# 무결성 검증
assert len(submission) == len(test), 'row count mismatch'
assert submission['pred_label'].isin(['POSITIVE','NEGATIVE']).all(), 'label format error'
assert submission['row_id'].is_unique, 'row_id 중복'

submission.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8')
print(f"saved: {SUBMISSION_PATH}  ({len(submission):,} rows)")
print(submission.head())


---

## 15. final_pipeline.pkl Save (재현/재로드)

### 15.1 저장 구조
모든 fit된 component를 dict 로 묶어 단일 pkl 저장. 이후 동일 환경에서 test 만 다시 inference 가능.


In [ ]:
pipeline = {
    'tfidf_word_vec': vec_w_full,
    'tfidf_word_clf': clf_w_full,
    'tfidf_char_vec': vec_c_full,
    'tfidf_char_clf': clf_c_full,
    'emb_e5_scaler':  sc_e5_full,
    'emb_e5_clf':     clf_e_full,
    'emb_kr_scaler':  sc_kr_full,
    'emb_kr_clf':     clf_k_full,
    'meta_scaler':    meta_scaler,
    'meta_clf':       meta_clf,
    'tau_final':      float(tau_final),
    'meta_cols':      META_COLS,
    'config': {
        'random_state':       SEED,
        'embedding_models':   list(EMB_MODELS.values()),
        'e5_prefix':          'query: ',
        'oof_mcc_final':      float(oof_mcc_final),
        'oof_mcc_default':    float(mcc_default),
        'tau_star_argmax':    float(tau_star),
        'tau_star_mcc':       float(mcc_star),
        'sklearn_version':    __import__('sklearn').__version__,
    },
}

PIPE_PATH = DATA_DIR / 'final_pipeline.pkl'
with open(PIPE_PATH, 'wb') as f:
    pickle.dump(pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = PIPE_PATH.stat().st_size / 1e6
print(f"saved: {PIPE_PATH}  ({size_mb:.1f} MB)")

# Reload sanity
with open(PIPE_PATH, 'rb') as f:
    loaded = pickle.load(f)
print(f"\nReload OK. keys: {list(loaded.keys())[:5]}...")
print(f"  tau_final={loaded['tau_final']:.4f}")
print(f"  oof_mcc_final={loaded['config']['oof_mcc_final']:.4f}")


saved: /content/drive/MyDrive/비정형 데이터 처리/final_pipeline.pkl  (66.6 MB)

Reload OK. keys: ['tfidf_word_vec', 'tfidf_word_clf', 'tfidf_char_vec', 'tfidf_char_clf', 'emb_e5_scaler']...
  tau_final=0.5000
  oof_mcc_final=0.7636


---

## 16. Discussion & Leaderboard Strategy

### 16.1 결과 요약
| Stage | OOF MCC | 메모 |
|---|---|---|
| Drill-05 baseline (LB) | 0.7570 | TF-IDF stacking, τ=0.5 |
| (B) emb only ensemble | ~0.75 | embedding 단독 ceiling |
| (D) 4-base avg | 측정값 위 참조 | diversity 효과 |
| (E) Meta stacking τ=0.5 | 측정값 위 참조 | non-linear weighting |
| (F) Meta + τ* | 측정값 위 참조 | threshold tuning gain |

### 16.2 CV-LB Gap 전망
- Drill-05 실측: Val 0.7456 → LB 0.7570 (+0.011, train이 더 어려움).
- Drill-06 OOF → LB 예상: 동일 분포 가정 시 **OOF + 0.005~0.015**.
- 즉, OOF 0.770 → LB ~0.775~0.785 기대.

### 16.3 Submission Strategy
1. **Primary**: 본 pipeline 결과 (final_pipeline.pkl 기반 submission.csv).
2. **Safety backup** (가능 시): τ=0.5 버전.
3. 절대 금지: LB 보고 weight/threshold 재조정 → public LB overfit 직행.

### 16.4 한계 & 후속 개선
- Embedding fine-tuning 미수행 (의도적 — overfit 위험 + 본 과제 가이드).
- LightGBM/XGBoost 미사용 (Drill-05 GBT meta learner LB drop 교훈 반영).
- 후속: 더 많은 embedding 모델 ensemble (bge-m3, e5-large) — 시간/메모리 허용 시.

### 16.5 재현성 체크리스트
- [x] random_state=42 모든 stochastic 연산에 고정
- [x] vectorizer/scaler fold 외부에서만 fit (leakage 차단)
- [x] threshold OOF 에서만 sweep
- [x] embedding caching (재현 시 동일 vector)
- [x] sklearn version 기록 (pkl config)
